# **ENTROPY**

In [37]:
import numpy as np

In [38]:
y=np.array([1,0,0,1,0,1,0,1])

In [39]:
def entropy(var):
    N=var.shape[0]
    values,counts= np.unique(var , return_counts=True)
    ent =0.0
    for i in counts:
        p=i/N
        ent += (p * np.log2(p))

    return -ent

In [40]:
entropy(y)

1.0

In [41]:
X=np.array([1,1,1,1,1,1,1,1])

In [42]:
entropy(X)

-0.0

# **SPLIT DATA**

In [43]:
import pandas as pd

In [44]:
df=pd.read_csv("golf.csv")

In [45]:
df

,Outlook,Temperature,Humidity,Windy,Play
0,sunny,hot,high,False,no
1,sunny,hot,high,True,no
2,overcast,hot,high,False,yes
3,rainy,mild,high,False,yes
4,rainy,cool,normal,False,yes
5,rainy,cool,normal,True,no
6,overcast,cool,normal,True,yes
7,sunny,mild,high,False,no
8,sunny,cool,normal,False,yes
9,rainy,mild,normal,False,yes


In [46]:
def divide_data(data,feature):
    DATA={}
    feat_values=list(data[feature].value_counts().index)
    occurence= list(data[feature].value_counts())

    for val in feat_values:
        DATA[val]={'data':pd.DataFrame([],columns=data.columns),'len':0}

    for ix in range(data.shape[0]):
        val=data[feature].iloc[ix]
        row_to_add=data.iloc[[ix]]
        DATA[val]['data']=pd.concat([DATA[val]['data'],row_to_add])
        idx=feat_values.index(val)
        DATA[val]['len']= occurence[idx]

    return DATA

In [47]:
divide_data(df,'Windy')

{False: {'data':      Outlook Temperature Humidity  Windy Play
  0      sunny         hot     high  False   no
  2   overcast         hot     high  False  yes
  3      rainy        mild     high  False  yes
  4      rainy        cool   normal  False  yes
  7      sunny        mild     high  False   no
  8      sunny        cool   normal  False  yes
  9      rainy        mild   normal  False  yes
  12  overcast         hot   normal  False  yes,
  'len': 8},
 True: {'data':      Outlook Temperature Humidity Windy Play
  1      sunny         hot     high  True   no
  5      rainy        cool   normal  True   no
  6   overcast        cool   normal  True  yes
  10     sunny        mild   normal  True  yes
  11  overcast        mild     high  True  yes
  13     rainy        mild     high  True   no,
  'len': 6}}

In [48]:
divide_data(df,'Outlook')

{'sunny': {'data':    Outlook Temperature Humidity  Windy Play
  0    sunny         hot     high  False   no
  1    sunny         hot     high   True   no
  7    sunny        mild     high  False   no
  8    sunny        cool   normal  False  yes
  10   sunny        mild   normal   True  yes,
  'len': 5},
 'rainy': {'data':    Outlook Temperature Humidity  Windy Play
  3    rainy        mild     high  False  yes
  4    rainy        cool   normal  False  yes
  5    rainy        cool   normal   True   no
  9    rainy        mild   normal  False  yes
  13   rainy        mild     high   True   no,
  'len': 5},
 'overcast': {'data':      Outlook Temperature Humidity  Windy Play
  2   overcast         hot     high  False  yes
  6   overcast        cool   normal   True  yes
  11  overcast        mild     high   True  yes
  12  overcast         hot   normal  False  yes,
  'len': 4}}

In [49]:
divide_data(df,'Temperature')

{'mild': {'data':      Outlook Temperature Humidity  Windy Play
  3      rainy        mild     high  False  yes
  7      sunny        mild     high  False   no
  9      rainy        mild   normal  False  yes
  10     sunny        mild   normal   True  yes
  11  overcast        mild     high   True  yes
  13     rainy        mild     high   True   no,
  'len': 6},
 'hot': {'data':      Outlook Temperature Humidity  Windy Play
  0      sunny         hot     high  False   no
  1      sunny         hot     high   True   no
  2   overcast         hot     high  False  yes
  12  overcast         hot   normal  False  yes,
  'len': 4},
 'cool': {'data':     Outlook Temperature Humidity  Windy Play
  4     rainy        cool   normal  False  yes
  5     rainy        cool   normal   True   no
  6  overcast        cool   normal   True  yes
  8     sunny        cool   normal  False  yes,
  'len': 4}}

# **Information Gain**

In [50]:
def information_gain(data,feature):
    examples=data.shape[0]

    DATA=divide_data(data, feature)
    keys=DATA.keys()

    ent_of_children=0.0
    for key in keys:
        ent_of_children+=((DATA[key]['len']/examples) * entropy(DATA[key]['data']['Play']))

    info_gain=entropy(data['Play'])-ent_of_children
    return info_gain

In [51]:
information_gain(df ,'Outlook')

0.24674981977443933

In [52]:
information_gain(df ,'Temperature')

0.02922256565895487

In [53]:
information_gain(df ,'Windy')

0.04812703040826949

# **IMPLEMENTAION DECISION TREE**

In [54]:
#DECISION TREE
class DecisionTree:
    #constructor
    def __init__(self,depth=0,max_depth=5):
        #Creating Node
        self.children={}
        self.fkey=None
        self.max_depth=max_depth
        self.depth=depth
        self.target= None

    def train(self,data):
        features=['Outlook','Temperature','Windy','Humidity']

        info_gain=[]
        for feature in features:
            i_gain=information_gain(data, feature)
            info_gain.append(i_gain)
        #finding the best feature
        self.fkey=features[np.argmax(info_gain)]

        #Splitting the Data
        DATA=divide_data(data,self.fkey)

        #Giving a target label to the Node
        labels=list(data['Play'].value_counts().index)
        freq=list(data['Play'].value_counts().values)

        self.target= labels[np.argmax(freq)]


        ##  Stoping Condition
        have_data=0
        keys=DATA.keys()
        for key in keys:
            if DATA[key]['len']>0:
                have_data+=1
        # 1.If it is a pure node
        if have_data<2:
             return
        #2.Early Stop if you have reached max depth
        if (self.depth)>=self.max_depth:
            return

        print('\t'*self.depth + "Making tree with - ",self.fkey)

        #Recursively trsin child Node
        for key in keys:
            new_data= DATA[key]['data']
            self.children[key]=DecisionTree(depth=self.depth+1,max_depth=self.max_depth)
            self.children[key].train(new_data)

        return

    def predict(self,test):
        if self.children== {}:
            return self.target

        return self.children[test[self.fkey][0]].predict(test)



In [55]:
#Step 2:Explore Decision model
model=DecisionTree()

In [56]:
model.train(df)

Making tree with -  Outlook
	Making tree with -  Humidity
	Making tree with -  Windy


In [57]:
model.target

'yes'

In [58]:
model.fkey

'Outlook'

In [59]:
model.children

{'sunny': <__main__.DecisionTree at 0x1e1cdedd4b0>,
 'rainy': <__main__.DecisionTree at 0x1e1cdedc3a0>,
 'overcast': <__main__.DecisionTree at 0x1e1cdedb100>}

In [60]:
model.children['sunny']

In [61]:
model.children['sunny'].target

'no'

In [62]:
model.children['sunny'].fkey

'Humidity'

In [63]:
model.children['sunny'].children

{'high': <__main__.DecisionTree at 0x1e1cdedcf10>,
 'normal': <__main__.DecisionTree at 0x1e1cdedaf50>}

In [64]:
#Prediction
x_test=pd.DataFrame([['overcast','hot','high',False]],columns=list(df.columns.values[:-1]))

In [65]:
x_test

,Outlook,Temperature,Humidity,Windy
0,overcast,hot,high,False


In [66]:
model.predict(x_test)

'yes'